In [1]:
dataset = '/mnt3/auto-ekyc/idrecapture/2.1 D.Tamper Printed Test Plan/mykadfront/orig/*.jpg'

In [ ]:
import lmdb
import numpy as np
import cv2
from PIL import Image
import torch
from torch.utils.data import Dataset

class CachedIDFraudTorchDataset(Dataset):

    def __init__(self, df, lmdb_path, transform=None):

        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(df['label'].unique().tolist())
        self.lmdb_path = lmdb_path

        self.env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False
        )

        self.txn = self.env.begin()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        label = row['label']

        key = f"{idx:08}".encode()

        img_bytes = self.txn.get(key)

        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

    def label_counts(self, percentages=False):

        counts = self.df['label'].value_counts().sort_index()

        label_map = {1: "fraud", 0: "genuine"}
        counts.index = counts.index.map(lambda x: label_map.get(x, str(x)))

        if not percentages:
            return counts

        total = counts.sum()
        return (counts / total).rename("percentage")

In [5]:
import sys

sys.path.append("/home/jingjie/AutoTorch/src")

from data.idfraud.preprocessing import preprocess_csv

In [ ]:
batch = ['1.1 Genuine with Background Test Plan/cleaned_filtered_annotated_index_annotation_jpg_subject_mykadfront_test.csv']
preprocess_csv('ori,')




import lmdb
import cv2
from tqdm import tqdm

env = lmdb.open("/mnt4/fraud_images.lmdb", map_size=100*1024**3)

with env.begin(write=True) as txn:

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        path = row["path"]

        img = cv2.imread(path)
        img = cv2.resize(img, (512,512))

        success, encoded = cv2.imencode(".jpg", img)

        key = f"{idx:08}".encode()

        txn.put(key, encoded.tobytes())

In [ ]:
import torch
import torchvision.transforms.v2 as v2
from PIL import Image
import os

# === CACHE (run once) ===
cache_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((512, 512), antialias=True),
])

pil_img = Image.open("your_image.jpg")
tensor = cache_transform(pil_img)
torch.save(tensor.data.clone(), "cached.pt")

print(f"Cached: {os.path.getsize('cached.pt') / 1024:.2f} KB")  # ~769 KB

# === LOAD (training) ===
load_transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tensor = load_transform(torch.load("cached.pt", weights_only=True))
# tensor.shape: [3, 512, 512], dtype: float32